### Markdown 

Middleware provides a way to more tightly control what happens inside an agent . Middleware is useful for the folloing
- Tracking agent behavior with logging, analytics and debugging
- Transforming prompts , tool selection and output formatting
- Adding retries, fallbacks and termination logic 
- Applying rate limits, guard rails and PII Protection

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

### Summerization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context.Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 5), # trigger when the length of LLM response reaches 5
            keep=("messages", 2) # use the last two messages to save as history

        )
    ]

)

In [ ]:
### Run with thread id
# thread_id is a configurable identifier used by LangGraph checkpointers to associate graph state with a specific conversation or execution thread. 
# When a graph is invoked with the same thread_id, LangGraph retrieves the previously checkpointed state and continues from it, enabling conversational memory and long-running workflows.
config={"configurable":{"thread_id":"test-1"}}

In [ ]:
# Alternative test data
questions=[
    " what is 2+2?",
    " what is 2%2?",
    " what is 2+10?",
    " what is 5+2?",
    " what is 12*2?",
    " what is 72+2?",
    " what is 92+27?",

]

from langchain_core.messages import HumanMessage

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")


In [ ]:
from langchain_core.tools import tool

@tool
def search_hotels(city:str)->str:
    """Search hotels - retuens long response to use more tokens"""

    return f"""Hotels in city{c}
    1. Grand Hotel
    2. Novetel
    3.Holiday Inn
    4. Budget Stay"""

### Token Size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent2 = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 2500), # trigger when the number of tokens>0500
            keep=("tokens", 100) # use the last two messages to save as history

        )
    ]

)

In [ ]:
### Run with thread id
# thread_id is a configurable identifier used by LangGraph checkpointers to associate graph state with a specific conversation or execution thread. 
# When a graph is invoked with the same thread_id, LangGraph retrieves the previously checkpointed state and continues from it, enabling conversational memory and long-running workflows.
config={"configurable":{"thread_id":"test-2"}}

In [ ]:
# Token count util

def count_token(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars / 4 # 4 chars = 1 token

In [ ]:
cities=["Paris", "London", " Tokyo", "Dubai", "Hyderabad"]

from langchain_core.messages import HumanMessage

for c in cities:
    response = agent.invoke({"messages":[HumanMessage(content=f"Find  hotels in the city{c}")]}, config)
    tokens = count_token(response["messages"])
    print(f"{c} : {tokens} tokens")
    print(f"{(response['messages'])}")

​
### Human-in-the-loop
- Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [ ]:
agent3 = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    tools=[read_email_tool ,send_email_tool],
    middleware=[
        
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]

)

In [ ]:
config={"configurable":{"thread_id":"test_approve"}}
from langchain_core.messages import HumanMessage


# Step1: Request
result = agent3.invoke({"messages":[HumanMessage(content=f"Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?")]}, config)
result


In [ ]:
# Step 2: Approve

from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused Approving...")

    result=agent3.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"approve"
                    }
                ]
            }
        ),
        config=config
    )

    print (f"Result:{result['messages'][-1].content}")


### Reject

In [ ]:
config={"configurable":{"thread_id":"test_reject"}}
from langchain_core.messages import HumanMessage


# Step1: Request
result = agent3.invoke({"messages":[HumanMessage(content=f"Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?")]}, config)
result


In [ ]:
# Step 2: Reject

from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused Approving...")

    result=agent3.invoke(
        Command(
            resume={
                "decisions":[
                    {
                        "type":"reject"
                    }
                ]
            }
        ),
        config=config
    )

    print (f"Result:{result['messages'][-1].content}")

### Edit

In [ ]:
config={"configurable":{"thread_id":"test_edit"}}
from langchain_core.messages import HumanMessage


# Step1: Request
result = agent3.invoke({"messages":[HumanMessage(content=f"Send Email to johndoe@example.com with Subject 'Hello' and body 'How Are you?")]}, config)
result


In [ ]:
# Step 2: Edit

from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused Approving...")

    result = agent3.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "someone@example.com",
                            "subject": "Updated subject",
                            "body": "Updated email body"
                        }
                    }
                }
            ]
        }
    ),
    config )

    print (f"Result:{result['messages'][-1].content}")

In [ ]:
result